In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
from torchinfo import summary
from of_transformer import OfTransformer
from net_utils import *
from data_utils import *

from sklearn.model_selection import train_test_split

import numpy as np
import uproot
import awkward as ak

from tqdm.notebook import tqdm

In [5]:
print(torch.cuda.is_available())
cuda_id = torch.cuda.current_device()
print(cuda_id)
print(torch.cuda.get_device_name(cuda_id))

True
0
NVIDIA A100-SXM4-80GB


Start by loading a single file for the MG sample using uproot. We will try to distinguish these events against themselves for now.

In [6]:
# Load mc
f_mc = uproot.open('/global/cfs/cdirs/m3246/ZjetOmnifold/data/slimmed_files/WithTracks_ZjetOmnifold_May19_MGPy8FxFxRew_syst_train.root')
tree_mc = f_mc['OmniTree']
tree_mc.show(name_width=50)

name                                               | typename                 | interpretation                
---------------------------------------------------+--------------------------+-------------------------------
weight                                             | float                    | AsDtype('>f4')
pass190                                            | int32_t                  | AsDtype('>i4')
truth_pass190                                      | int32_t                  | AsDtype('>i4')
weight_mc                                          | float                    | AsDtype('>f4')
prw                                                | float                    | AsDtype('>f4')
pass190_syst_ID_Up                                 | int32_t                  | AsDtype('>i4')
pass190_syst_ID_Down                               | int32_t                  | AsDtype('>i4')
pass190_syst_MS_Up                                 | int32_t                  | AsDtype('>i4')
pass190_syst_MS_Do

In [7]:
f_pd = uproot.open('/global/cfs/cdirs/m3246/ZjetOmnifold/data/slimmed_files/WithTracks_ZjetOmnifold_Aug5_PseudoDataSRew_Dec15.root')
tree_pd = f_pd['OmniTree']
tree_pd.show(name_width=50)

name                                               | typename                 | interpretation                
---------------------------------------------------+--------------------------+-------------------------------
weight                                             | float                    | AsDtype('>f4')
pass190                                            | int32_t                  | AsDtype('>i4')
truth_pass190                                      | int32_t                  | AsDtype('>i4')
weight_mc                                          | float                    | AsDtype('>f4')
prw                                                | float                    | AsDtype('>f4')
pT_ll                                              | float                    | AsDtype('>f4')
pT_l1                                              | float                    | AsDtype('>f4')
pT_l2                                              | float                    | AsDtype('>f4')
eta_l1            

Next we need to build torch tensors from the data. One open question is how to handle the muon kinematics, since they are distinct from the tracks. For now I will include the muon information in the same way as the tracks, but this should be changed in the future. Probably the best way is to include a one-hot encoded input, which tells the network where the 3-vector is a muon, track from 1st jet, track from 2nd jet, etc.

In [8]:
# Pass 190 flags
pass190_mc = ak.to_numpy(tree_mc['pass190'].array())
print("We have a fracion {} of good events in mc".format(np.sum(pass190_mc) / len(pass190_mc)))
pass190_pd = ak.to_numpy(tree_pd['pass190'].array())
print("We have a fracion {} of good events in pseudodata".format(np.sum(pass190_pd) / len(pass190_pd)))

We have a fracion 0.8144455536320278 of good events in mc
We have a fracion 1.0 of good events in pseudodata


In [9]:
# Load MC kinematics
mc_kinematics, mc_mask = get_kinematics(tree_mc, filter=pass190_mc, max_tracks=None)
print(mc_kinematics.shape)
print(mc_mask.shape)


(1160150, 3, 294)
(1160150, 1, 294)


In [10]:
# Load pseudodata kinematics
pd_kinematics, pd_mask = get_kinematics(tree_pd, filter=pass190_pd, max_tracks=mc_kinematics.shape[2]-2)
print(pd_kinematics.shape)
print(pd_mask.shape)

Warning! Track kinematics shape does not match muon kinematics shape!
(235917, 3, 294)
(235917, 1, 294)


In [11]:
# # Make one-hot encoding identifying whether the object is a muon or a track
# is_muon = np.concatenate([np.ones((kinematics.shape[0], 2)), np.zeros((kinematics.shape[0], track_kinematics.shape[2]))], axis=1)
# is_track = np.concatenate([np.zeros((kinematics.shape[0], 2)), np.ones((kinematics.shape[0], track_kinematics.shape[2]))], axis=1)
# one_hot = np.stack([is_muon, is_track], axis=1)
# print(one_hot.shape)

Next we need to load the weights. For MC this is easy. For pseudodata we need to take the missing track events into account.

In [12]:
# MC weights
mc_weights = ak.to_numpy(tree_mc['weight'].array())
mc_weights /= np.mean(mc_weights)
mc_weights = np.expand_dims(mc_weights[pass190_mc == 1], axis=1)
print(mc_weights.shape)

(1160150, 1)


In [13]:
# Pseudodata weights
pd_weights = ak.to_numpy(tree_pd['weight'].array())
pd_weights /= np.mean(pd_weights)
pd_weights = np.expand_dims(pd_weights[pass190_pd == 1], axis=1)

# Drop 11k weights at the end since we don't have tracks for those events
pd_weights = pd_weights[:pd_kinematics.shape[0]]

print(pd_weights.shape)

(235917, 1)


Finally the labels. We will call pseudodata signal, and MC background.

In [14]:
# Build labels
mc_labels = np.zeros((mc_kinematics.shape[0], 1), dtype=np.float32)
pd_labels = np.ones((pd_kinematics.shape[0], 1), dtype=np.float32)

In [15]:
# Concatenate MC and pseudodata together, taking as many MC events as there are pseudodata events
kinematics = np.concatenate([mc_kinematics[:pd_kinematics.shape[0],...], pd_kinematics], axis=0)
mask = np.concatenate([mc_mask[:pd_mask.shape[0],...], pd_mask], axis=0)
weights = np.concatenate([mc_weights[:pd_weights.shape[0],...], pd_weights], axis=0)
labels = np.concatenate([mc_labels[:pd_labels.shape[0],...], pd_labels], axis=0)

In [16]:
# Make train test split
kinematics_train, kinematics_test, labels_train, labels_test, mask_train, mask_test, weights_train, weights_test = train_test_split(kinematics, labels, mask, weights, test_size=0.2, random_state=42)
print(kinematics_train.shape, kinematics_test.shape, labels_train.shape, labels_test.shape, mask_train.shape, mask_test.shape, weights_train.shape, weights_test.shape)

(377467, 3, 294) (94367, 3, 294) (377467, 1) (94367, 1) (377467, 1, 294) (94367, 1, 294) (377467, 1) (94367, 1)


In [17]:
# Convert to torch tensors
device = 'cuda:0'
kinematics_train = torch.tensor(kinematics_train, dtype=torch.float32).to(device)
kinematics_test = torch.tensor(kinematics_test, dtype=torch.float32).to(device)
labels_train = torch.tensor(labels_train, dtype=torch.float32).to(device)
labels_test = torch.tensor(labels_test, dtype=torch.float32).to(device)
mask_train = torch.tensor(mask_train, dtype=torch.float32).to(device)
mask_test = torch.tensor(mask_test, dtype=torch.float32).to(device)
weights_train = torch.tensor(weights_train, dtype=torch.float32).to(device)
weights_test = torch.tensor(weights_test, dtype=torch.float32).to(device)

In [18]:
# Build a pytorch dataset
train_dataset = torch.utils.data.TensorDataset(kinematics_train, labels_train, mask_train, weights_train)
test_dataset = torch.utils.data.TensorDataset(kinematics_test, labels_test, mask_test, weights_test)

In [19]:
# Build pytorch data loaders
batch_size = 32
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=True)

Data is ready. Next we need to build a model!

In [26]:
part = OfTransformer(
    3,
    num_classes=1,
    trim=True,
    fc_params=[(100, 0.0), (100, 0.0), (1, 0.5)],
    pair_embed_dims=None
)
next(part.parameters()).device

device(type='cpu')

In [27]:
# Copy model to GPU
part.to('cuda:0')
next(part.parameters()).device

device(type='cuda', index=0)

In [28]:
summary(part, input_shape=kinematics_train.shape[1:])

Layer (type:depth-idx)                                       Param #
OfTransformer                                                128
├─SequenceTrimmer: 1-1                                       --
├─Embed: 1-2                                                 --
│    └─BatchNorm1d: 2-1                                      6
│    └─Sequential: 2-2                                       --
│    │    └─LayerNorm: 3-1                                   6
│    │    └─Linear: 3-2                                      512
│    │    └─GELU: 3-3                                        --
│    │    └─LayerNorm: 3-4                                   256
│    │    └─Linear: 3-5                                      66,048
│    │    └─GELU: 3-6                                        --
│    │    └─LayerNorm: 3-7                                   1,024
│    │    └─Linear: 3-8                                      65,664
│    │    └─GELU: 3-9                                        --
├─ModuleList: 1-3      

In [29]:
# Practice forward pass
first_event = kinematics_train[0:1,...]
first_mask = mask_train[0:1,...]
with torch.no_grad():
    out = part(first_event, mask=first_mask)
print(out)


tensor([[-0.6506]], device='cuda:0')


Now train the model. Define and optimizer and a loss function

In [30]:
criterion = torch.nn.BCEWithLogitsLoss(reduction='none')
optimizer = torch.optim.AdamW(part.parameters(), lr=1e-3)

And train for a few epochs

In [31]:
epochs = 10

for epoch in range(epochs):  # loop over the dataset multiple times

    print("Epoch {}".format(epoch + 1))

    loop_obj = tqdm(train_loader)
    running_loss = 0.0
    for i, batch in enumerate(loop_obj):

        # Unpack batch 
        batch_kinematics, batch_labels, batch_mask, batch_weights = batch

        # Zero gradients
        optimizer.zero_grad()

        # Forward pass, compute loss, backward pass, optimizer step
        out = part(batch_kinematics, mask=batch_mask)
        loss = criterion(out, batch_labels)
        loss = loss * batch_weights
        loss.mean().backward()
        optimizer.step()

        # Print stats
        running_loss += loss.mean().item()
        if i % 100 == 99:    # print every 100 mini-batches
            # print(f'Epoch {epoch + 1}, mini-batch {i + 1}: loss {running_loss / 100:.3f}')
            loop_obj.set_postfix({'loss': running_loss / 100})
            running_loss = 0.0

Epoch 1


  0%|          | 0/11795 [00:00<?, ?it/s]

Epoch 2


  0%|          | 0/11795 [00:00<?, ?it/s]

Epoch 3


...

  0%|          | 0/11795 [00:00<?, ?it/s]

Epoch 9


  0%|          | 0/11795 [00:00<?, ?it/s]

Epoch 10


  0%|          | 0/11795 [00:00<?, ?it/s]